In [ ]:
import pandas as pd

In [ ]:
import pingouin as pg

In [ ]:
with open("results.json", encoding="utf-8") as f:
    results = pd.read_json(f)

In [ ]:
results

In [ ]:
results["header"][0]

In [ ]:
results["rows"][0]

In [ ]:
results_df = pd.DataFrame(results["rows"][0], columns=results["header"][0])

In [ ]:
results_df

In [ ]:
results_df["roundCount"].value_counts()

In [ ]:
results_df["player1Id"].value_counts()

In [ ]:
results_df["gameConfigPlayer1"].value_counts() 

In [ ]:
results_df["gameConfigPlayer2"].value_counts()

In [ ]:
results_df["player2Id"].value_counts()

In [ ]:
results_df["player1Id"].concat(results_df["player2Id"])

In [ ]:
results_df["status"].value_counts()

In [ ]:
results_df_finished = results_df[results_df["status"] != "in_progress"]

In [ ]:
pd.concat([results_df_finished["player1Id"], results_df_finished["player2Id"]], ignore_index=True).value_counts()

In [ ]:
results_df_finished

In [ ]:
results_df_finished["roundCountFinished"] = results_df_finished["roundCount"].apply(lambda x: 16 if int(x) > 16 else int(x))

In [ ]:
results_df_finished["roundCountFinished"].value_counts()

In [ ]:
results_df_finished["roundCount"].value_counts()

In [ ]:
results_df_finished["roundCountFinished"].hist()

In [ ]:
results_df_finished.groupby("gameConfigPlayer1")["roundCountFinished"].value_counts()

In [ ]:
results_df_finished.groupby("gameConfigPlayer1")["roundCountFinished"].hist()

In [ ]:
results_df_finished.groupby("gameConfigPlayer1")["roundCountFinished"].describe()

In [ ]:
results_df_finished["statusFinished"] = results_df_finished["roundCount"].apply(lambda x: "won" if int(x) < 16 else "lost")

In [ ]:
results_df_finished.groupby("gameConfigPlayer1")["status"].value_counts()

In [ ]:
results_df_finished.groupby("gameConfigPlayer1")["statusFinished"].value_counts(normalize=True)

In [ ]:
results_df_finished.groupby("gameConfigPlayer1")["statusFinished"].value_counts(normalize=True)

In [ ]:
pg.kruskal(results_df_finished, "roundCountFinished", between="gameConfigPlayer1")

In [ ]:
pg.pairwise_tests(results_df_finished, "roundCountFinished", between="gameConfigPlayer1", parametric=False, padjust="bonf")

In [ ]:
results_df_finished.groupby("gameConfigPlayer1").hist(column="roundCountFinished")

In [ ]:
results_df_finished.groupby("gameConfigPlayer1").boxplot(column="roundCountFinished")

In [ ]:
player_id_array = ["DvplG2Kz", "IfM2wWHp", "1zx9ju2S", "BVSUju5U", "lW70ICul", "OZ52imkd", "GorQHcOB", "8PSrn9JD", "u5LgXigL",
                  "1Ax1SPCe", "l8ND7njk", "sfiXibsa", "PvUHGrDy", "MO2pWpjE", "gsbTiZwO", "gDnhDODC", "Ska8rixW", "6DbWtL5r",
                  "TCFqdHBb", "wQhT1jrv"]


In [ ]:
len(player_id_array)

In [ ]:
results_df

# New filtering

In [ ]:
player_id_array = ["DvplG2Kz", "IfM2wWHp", "1zx9ju2S", "BVSUju5U", "lW70ICul", "OZ52imkd", "GorQHcOB", "8PSrn9JD", "u5LgXigL",
                  "1Ax1SPCe", "l8ND7njk", "sfiXibsa", "PvUHGrDy", "MO2pWpjE", "gsbTiZwO", "gDnhDODC", "Ska8rixW", "6DbWtL5r",
                  "TCFqdHBb", "wQhT1jrv"]
results_df["roundCount"] = results_df["roundCount"].astype(int)
filtered_df = results_df[results_df["player1Id"].isin(player_id_array)]
filtered_df = results_df[(results_df["roundCount"] > 2) & (results_df["status"] != "in_progress")]

In [ ]:
filtered_df

In [ ]:
filtered_df["status"].value_counts()

In [ ]:
filtered_df["statusCorrected"] = filtered_df.apply(lambda x: "lost" if x["roundCount"] > 16 else x["status"], axis=1)

In [ ]:
filtered_df_onlywon = filtered_df[filtered_df["statusCorrected"] == "won"]

In [ ]:
filtered_df_onlywon

In [ ]:
filtered_df_onlywon["statusCorrected"].value_counts()

## What is the success rate?

In [ ]:
filtered_df.groupby("gameConfigPlayer1")["statusCorrected"].value_counts(normalize=True)

In [ ]:
filtered_df.groupby("gameConfigPlayer1").describe()

In [ ]:
filtered_df.groupby("gameConfigPlayer2").describe()

In [ ]:
filtered_df.groupby("trueGameConfig").describe()

In [ ]:
filtered_df.groupby("trueGameConfig")["statusCorrected"].value_counts(normalize=True)

In [ ]:
filtered_df[filtered_df["trueGameConfig"] == "human_vs_bot"]["statusCorrected"].apply(lambda x: x=="won")

In [ ]:
pg.mwu(filtered_df[filtered_df["trueGameConfig"] == "human_vs_bot"]["statusCorrected"].apply(lambda x: x=="won"), filtered_df[filtered_df["trueGameConfig"] == "human_vs_human"]["statusCorrected"].apply(lambda x: x=="won"))

In [ ]:
pg.chi2_independence(filtered_df[filtered_df["trueGameConfig"] == "human_vs_bot"]["statusCorrected"].apply(lambda x: x=="won"), filtered_df[filtered_df["trueGameConfig"] == "human_vs_human"]["statusCorrected"].apply(lambda x: x=="won"))

## What is the convergence time?

In [ ]:
filtered_df_onlywon.groupby("gameConfigPlayer1").boxplot(column="roundCount")

In [ ]:
filtered_df_onlywon.groupby("gameConfigPlayer1").describe()

In [ ]:
pg.kruskal(filtered_df_onlywon, "roundCount", between="gameConfigPlayer1")

In [ ]:
pg.pairwise_tests(filtered_df_onlywon, "roundCount", between="gameConfigPlayer1", parametric=False, padjust="bonf")

In [ ]:
pg.pairwise_tests(filtered_df_onlywon, "roundCount", between="trueGameConfig", parametric=False, padjust="bonf")

In [ ]:
filtered_df_onlywon.groupby("trueGameConfig").boxplot(column="roundCount")

In [ ]:
filtered_df_onlywon.groupby("trueGameConfig").describe()

In [ ]:
filtered_df_onlywon.groupby("trueGameConfig").hist()

In [ ]:
filtered_df.groupby("gameConfigPlayer1").boxplot(column="roundCount")

## Conceptual linking

In [ ]:
with open("results.csv") as f:
    results2_df = pd.read_csv(f)

In [ ]:
results2_df

In [ ]:
results2_df["roundCount"] = results2_df["roundCount"].astype(int)
filtered_df2 = results2_df[results2_df["player1Id"].isin(player_id_array)]
filtered_df2 = results2_df[(results2_df["roundCount"] > 2) & (results2_df["status"] != "in_progress")]

In [ ]:
filtered_df2

In [ ]:
filtered_df2["gameConfigPlayer1"].value_counts()

In [ ]:
filtered_df["gameConfigPlayer1"].value_counts()

In [ ]:
filtered_df["gameConfigPlayer2"].value_counts()

In [ ]:
filtered_df2.columns

In [ ]:
filtered_df2["conceptual_linking_score_my"]

In [ ]:
import ast

In [ ]:
filtered_df2["conceptual_linking_score_my_mean"] = filtered_df2["conceptual_linking_score_my"].apply(lambda x: pd.Series(ast.literal_eval(x)).mean())

In [ ]:
filtered_df2["conceptual_linking_score_opponent_mean"] = filtered_df2["conceptual_linking_score_opponent"].apply(lambda x: pd.Series(ast.literal_eval(x)).mean())

In [ ]:
pg.kruskal(filtered_df2, "conceptual_linking_score_opponent_mean", between="gameConfigPlayer1")

In [ ]:
pg.pairwise_tests(filtered_df2, "conceptual_linking_score_opponent_mean", between="gameConfigPlayer1", parametric=False, padjust="bonf")

In [ ]:
filtered_df2["trueGameConfig"] = filtered_df2["gameConfigPlayer1"].apply(lambda x: "human_vs_bot" if "human_vs_bot" in x else "human_vs_human")

In [ ]:
pg.pairwise_tests(filtered_df2, "conceptual_linking_score_opponent_mean", between="trueGameConfig", parametric=False, padjust="bonf")

## Self-reported opponent player strategy